

# Handling Categorical Values

Most Machine Learning algorithms work best with numerical data. However, real-world datasets often contain categorical or text attributes.

Scikit-Learn provides encoders to convert categorical values into numerical representations.

We will use the `ocean_proximity` column from the California housing dataset as an example.

---

## 1. Categorical Attributes

Categorical attributes contain a limited set of possible values rather than arbitrary text.

For example, the `ocean_proximity` column may contain categories such as:

- `"NEAR BAY"`
- `"INLAND"`
- `"ISLAND"`
- `"<1H OCEAN"`
- `"NEAR OCEAN"`

First, select the categorical column:

```python
housing_cat = housing[["ocean_proximity"]]

housing_cat.head()
```

**Why use double square brackets?**

`housing[["ocean_proximity"]]` returns a DataFrame with one column, which is the expected input format for these encoders.

---

## 2. Ordinal Encoding

**Ordinal encoding** converts each category into a numerical code.

Scikit-Learn provides `OrdinalEncoder` for this purpose.

```python
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()

housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)

housing_cat_encoded[:5]
```

The output is a two-dimensional NumPy array containing numerical category codes.

### Check the Categories

```python
ordinal_encoder.categories_
```

Example output:

```text
[
    array(['<1H OCEAN', 'INLAND', 'ISLAND',
           'NEAR BAY', 'NEAR OCEAN'], dtype=object)
]
```

The encoder assigns a numerical code to each category based on its position in the learned category list.

For this example, the mapping is:

| Category | Encoded Value |
|---|---:|
| `<1H OCEAN` | 0 |
| `INLAND` | 1 |
| `ISLAND` | 2 |
| `NEAR BAY` | 3 |
| `NEAR OCEAN` | 4 |

**Important limitation:**

Ordinal encoding can imply that one category is greater than or less than another.

For example, the model might interpret `4` as greater than `1`, even though `"NEAR OCEAN"` and `"INLAND"` do not have a meaningful numerical order.

Therefore, ordinal encoding is generally appropriate when categories have a genuine order, such as:

- `"Low"` → 0
- `"Medium"` → 1
- `"High"` → 2

---

## 3. One-Hot Encoding

For categorical features without a meaningful order, **one-hot encoding** is often a better choice.

It creates a separate binary column for each category.

```python
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()

housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
```

### How Does It Work?

Suppose the categories are:

```text
INLAND
NEAR BAY
NEAR OCEAN
```

One-hot encoding creates three columns:

| INLAND | NEAR BAY | NEAR OCEAN |
|---:|---:|---:|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

Each row contains a `1` for its category and `0` for the others.

This avoids assigning an artificial numerical ranking to the categories.

### Convert the Sparse Matrix to a NumPy Array

By default, `OneHotEncoder` commonly returns a sparse matrix, which efficiently stores data containing many zeros.

To convert it into a regular, dense NumPy array:

```python
housing_cat_1hot.toarray()
```

### Get a Dense Array Directly

In newer Scikit-Learn versions, use `sparse_output=False`:

```python
cat_encoder = OneHotEncoder(sparse_output=False)

housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
```

For older Scikit-Learn versions, the equivalent argument is `sparse=False`.

### Check the Category Order

```python
cat_encoder.categories_
```

This shows the categories used by the encoder and the corresponding order of the generated columns.

---

## 4. Ordinal Encoding vs. One-Hot Encoding

| Feature | OrdinalEncoder | OneHotEncoder |
|---|---|---|
| Best suited for | Ordered categories | Unordered categories |
| Representation | One numerical code per category | One binary column per category |
| Output | 2D NumPy array | Sparse or dense matrix |
| Artificial order | May imply an order | Avoids numerical ranking |
| Example | Low, Medium, High | INLAND, NEAR BAY, ISLAND |

---

## 5. Summary

- **Categorical attributes** contain a limited set of possible values.
- `OrdinalEncoder` converts categories into numerical codes.
- Use ordinal encoding when categories have a meaningful order.
- `OneHotEncoder` creates a separate binary column for each category.
- One-hot encoding is often preferable for unordered categories.
- Sparse matrices save memory when most values are zero.
- Use `cat_encoder.categories_` to inspect the learned categories.

**Key takeaway:** Choose the encoding method according to the meaning of your categorical data, not just because a Machine Learning algorithm requires numerical input.


In [2]:
import pandas as pd 

In [3]:
import numpy as np
import matplotlib.pyplot as plt


In [4]:
df = pd.read_csv("housing.csv")
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [5]:
df['income_category'] = pd.cut(df['median_income'],bins=[0,1.5,3.0,4.5,6.0,np.inf],labels = [1,2,3,4,5])

In [6]:
from sklearn.model_selection import StratifiedShuffleSplit

In [7]:
split = StratifiedShuffleSplit(n_splits = 1 , test_size = 0.2 , random_state = 42)

In [8]:
for train_index , test_index in split.split(df,df["income_category"]):

    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]

In [9]:
for sett in (strat_train_set,strat_test_set):
    sett.drop("income_category" , axis = 1 , inplace = True)

In [10]:
df = strat_train_set.copy()

In [11]:
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736,72100.0,INLAND
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373,279600.0,NEAR OCEAN
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750,82700.0,INLAND
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264,112500.0,NEAR OCEAN
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964,238300.0,<1H OCEAN
...,...,...,...,...,...,...,...,...,...,...
15174,-117.07,33.03,14.0,6665.0,1231.0,2026.0,1001.0,5.0900,268500.0,<1H OCEAN
12661,-121.42,38.51,15.0,7901.0,1422.0,4769.0,1418.0,2.8139,90400.0,INLAND
19263,-122.72,38.44,48.0,707.0,166.0,458.0,172.0,3.1797,140400.0,<1H OCEAN
19140,-122.70,38.31,14.0,3155.0,580.0,1208.0,501.0,4.1964,258100.0,<1H OCEAN


In [12]:
housing = strat_train_set.drop("median_house_value",axis = 1)
housing_labels = strat_train_set["median_house_value"].copy()

In [13]:
housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736,INLAND
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373,NEAR OCEAN
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750,INLAND
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264,NEAR OCEAN
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964,<1H OCEAN
...,...,...,...,...,...,...,...,...,...
15174,-117.07,33.03,14.0,6665.0,1231.0,2026.0,1001.0,5.0900,<1H OCEAN
12661,-121.42,38.51,15.0,7901.0,1422.0,4769.0,1418.0,2.8139,INLAND
19263,-122.72,38.44,48.0,707.0,166.0,458.0,172.0,3.1797,<1H OCEAN
19140,-122.70,38.31,14.0,3155.0,580.0,1208.0,501.0,4.1964,<1H OCEAN


In [14]:
housing_labels

12655     72100.0
15502    279600.0
2908      82700.0
14053    112500.0
20496    238300.0
           ...   
15174    268500.0
12661     90400.0
19263    140400.0
19140    258100.0
19773     62700.0
Name: median_house_value, Length: 16512, dtype: float64

In [15]:
from sklearn.impute import SimpleImputer

In [16]:
imputer =  SimpleImputer(strategy="median")

In [17]:
housing_num = housing.select_dtypes(include=[np.number])
imputer.fit(housing_num)

SimpleImputer(strategy='median')

In [18]:
imputer.statistics_

array([-118.51   ,   34.26   ,   29.     , 2119.     ,  433.     ,
       1164.     ,  408.     ,    3.54155])

In [19]:
x = imputer.transform(housing_num)

In [20]:
x

array([[-1.2146e+02,  3.8520e+01,  2.9000e+01, ...,  2.2370e+03,
         7.0600e+02,  2.1736e+00],
       [-1.1723e+02,  3.3090e+01,  7.0000e+00, ...,  2.0150e+03,
         7.6800e+02,  6.3373e+00],
       [-1.1904e+02,  3.5370e+01,  4.4000e+01, ...,  6.6700e+02,
         3.0000e+02,  2.8750e+00],
       ...,
       [-1.2272e+02,  3.8440e+01,  4.8000e+01, ...,  4.5800e+02,
         1.7200e+02,  3.1797e+00],
       [-1.2270e+02,  3.8310e+01,  1.4000e+01, ...,  1.2080e+03,
         5.0100e+02,  4.1964e+00],
       [-1.2214e+02,  3.9970e+01,  2.7000e+01, ...,  6.2500e+02,
         1.9700e+02,  3.1319e+00]])

In [21]:
housing - pd.DataFrame(x,columns = housing_num.columns , index= housing_num.index)

,households,housing_median_age,latitude,longitude,median_income,ocean_proximity,population,total_bedrooms,total_rooms
12655,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0
15502,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0
2908,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0
14053,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0
20496,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
15174,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0
12661,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0
19263,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0
19140,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0


In [22]:
housing['ocean_proximity'] = df['ocean_proximity']

In [23]:
housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736,INLAND
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373,NEAR OCEAN
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750,INLAND
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264,NEAR OCEAN
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964,<1H OCEAN
...,...,...,...,...,...,...,...,...,...
15174,-117.07,33.03,14.0,6665.0,1231.0,2026.0,1001.0,5.0900,<1H OCEAN
12661,-121.42,38.51,15.0,7901.0,1422.0,4769.0,1418.0,2.8139,INLAND
19263,-122.72,38.44,48.0,707.0,166.0,458.0,172.0,3.1797,<1H OCEAN
19140,-122.70,38.31,14.0,3155.0,580.0,1208.0,501.0,4.1964,<1H OCEAN


In [24]:
set(housing['ocean_proximity'])

{'<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'}

In [25]:
from sklearn.preprocessing import OrdinalEncoder

In [26]:
ordinal_encoder = OrdinalEncoder()

In [27]:
housing_cat = ordinal_encoder.fit_transform(housing)

In [29]:
housing_cat

array([[2.390e+02, 5.690e+02, 2.800e+01, ..., 7.040e+02, 1.815e+03,
        1.000e+00],
       [6.620e+02, 5.500e+01, 6.000e+00, ..., 7.660e+02, 9.519e+03,
        4.000e+00],
       [4.810e+02, 2.730e+02, 4.300e+01, ..., 2.980e+02, 3.475e+03,
        1.000e+00],
       ...,
       [1.130e+02, 5.610e+02, 4.700e+01, ..., 1.700e+02, 4.239e+03,
        0.000e+00],
       [1.150e+02, 5.480e+02, 1.300e+01, ..., 4.990e+02, 6.403e+03,
        0.000e+00],
       [1.710e+02, 7.010e+02, 2.600e+01, ..., 1.950e+02, 4.107e+03,
        1.000e+00]])

In [31]:
housing_cat = pd.DataFrame(housing_cat ,columns = housing.columns , index= housing.index)

In [33]:
from sklearn.preprocessing import OneHotEncoder

In [35]:
ordinal_encoder = OneHotEncoder()

In [49]:
housing_cat = ordinal_encoder.fit_transform(housing)

In [51]:
housing_cat.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [55]:
housing_cat.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])